# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [2]:
LITE_MODE = True


load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [13]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [14]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 33816.434, Val Loss: 20379.973


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 15372.784, Val Loss: 18024.166


In [15]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [16]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$65 $40 $20 $22 $37 $180 $27 $69 $20 $52 $449 $99 $87 $188 $14 $16 $4 $12 $65 $34 $43 $27 $68 $26 $284 $280 $237 $23 $12 $44 $33 $153 $43 $34 $114 $281 $29 $112 $111 $43 $164 $96 $14 $104 $118 $40 $55 $43 $25 $25 $7 $61 $132 $29 $94 $108 $31 $123 $33 $33 $72 $0 $47 $10 $438 $81 $40 $291 $23 $247 $4 $21 $113 $107 $15 $57 $88 $37 $23 $51 $72 $118 $23 $50 $10 $49 $63 $141 $113 $109 $17 $67 $23 $8 $29 $103 $53 $28 $125 $174 $10 $63 $2 $40 $2 $64 $87 $252 $4 $107 $22 $44 $124 $54 $39 $161 $147 $45 $60 $33 $19 $218 $17 $13 $70 $25 $17 $208 $49 $56 $53 $138 $107 $15 $69 $20 $97 $46 $44 $43 $25 $156 $17 $137 $206 $61 $43 $307 $71 $8 $16 $197 $3 $69 $25 $142 $130 $18 $58 $14 $87 $9 $4 $22 $480 $13 $35 $14 $5 $32 $10 $16 $242 $42 $4 $15 $12 $9 $36 $114 $375 $10 $137 $24 $27 $71 $34 $36 $17 $9 $42 $51 $63 $39 $13 $17 $95 $13 $8 $10 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [17]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [18]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [19]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [20]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [21]:
gpt_4__1_nano(test[0])

'$180'

In [22]:
test[0].price

219.0

In [24]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $4 $30 $20 $20 $110 $6 $65 $6 $870 $413 $20 $10 $21 $1 $8 $41 $10 $40 $31 $59 $56 $65 $125 $212 $304 $355 $0 $151 $64 $30 $15 $10 $55 $35 $219 $90 $31 $36 $13 $175 $45 $20 $105 $70 $5 $17 $3 $65 $52 $20 $105 $125 $0 $147 $54 $8 $20 $52 $3 $116 $2 $41 $30 $66 $10 $90 $295 $25 $74 $16 $8 $20 $4 $5 $21 $126 $0 $3 $4 $0 $3 $10 $74 $11 $0 $32 $44 $30 $16 $8 $20 $5 $20 $2 $78 $1 $7 $20 $225 $50 $3 $21 $11 $150 $82 $10 $380 $18 $1 $10 $536 $119 $78 $54 $180 $0 $4 $94 $47 $29 $511 $30 $46 $0 $10 $5 $101 $29 $94 $29 $13 $65 $5 $85 $0 $85 $10 $53 $62 $16 $150 $0 $12 $124 $18 $5 $440 $15 $13 $4 $144 $17 $10 $4 $29 $101 $41 $20 $5 $661 $18 $3 $2 $240 $2 $502 $25 $0 $5 $5 $3 $120 $8 $42 $201 $3 $57 $36 $18 $546 $20 $150 $49 $70 $8 $83 $7 $10 $2 $5 $89 $30 $11 $50 $70 $10 $20 $21 $1 

In [25]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [27]:
evaluate(claude_opus_4_5, test, workers = 1)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $34 $25 $25 $0 $30 $64 $25 $6 $45 $214 $129 $5 $24 $49 $3 $11 $20 $20 $74 $16 $11 $40 $25 $33 $253 $206 $5 $90 $65 $10 $30 $70 $50 $25 $290 $30 $43 $39 $8 $154 $55 $10 $5 $70 $0 $5 $2 $65 $72 $28 $114 $325 $10 $37 $44 $6 $50 $48 $1 $46 $48 $61 $60 $279 $9 $50 $355 $35 $14 $19 $3 $70 $6 $20 $16 $75 $2 $2 $6 $10 $3 $5 $74 $14 $25 $22 $44 $40 $1 $3 $5 $5 $22 $1 $99 $4 $67 $120 $225 $10 $27 $3 $49 $59 $32 $12 $365 $1 $114 $30 $36 $5 $38 $4 $29 $9 $7 $24 $47 $4 $161 $10 $76 $0 $15 $4 $9 $30 $49 $119 $8 $39 $0 $25 $2 $55 $10 $33 $12 $16 $249 $30 $7 $44 $2 $15 $25 $85 $8 $6 $83 $31 $94 $1 $89 $26 $43 $35 $20 $40 $18 $8 $0 $41 $7 $58 $20 $0 $0 $9 $9 $170 $14 $59 $19 $2 $3 $4 $48 $155 $10 $250 $59 $24 $3 $53 $12 $25 $14 $5 $1 $10 $11 $70 $10 $9 $130 $21 $7 

In [28]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [29]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

  0%|          | 0/50 [00:00<?, ?it/s]

$10 $34 $15 $20 $0 $50 $84 $25 $14 $110 $186 $30 $5 $4 $49 $3 $21 $25 $11 $29 $4 $14 $25 $35 $117 $204 $275 $3 $101 $64 $0 $30 $0 $60 $15 $170 $6 $39 $16 $9 $160 $40 $15 $105 $20 $0 $2 $2 $75 $32 

In [30]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [31]:
evaluate(gemini_2__5_flash_lite, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $104 $15 $25 $30 $80 $69 $95 $9 $120 $438 $20 $25 $4 $4 $12 $71 $10 $140 $49 $9 $24 $10 $75 $132 $203 $145 $5 $251 $60 $15 $15 $35 $55 $35 $31 $90 $31 $14 $13 $150 $15 $10 $95 $120 $10 $17 $13 $75 $48 $30 $85 $125 $10 $67 $16 $8 $30 $22 $13 $14 $33 $51 $40 $79 $35 $60 $325 $10 $74 $15 $8 $130 $1 $5 $11 $126 $3 $8 $4 $0 $3 $5 $74 $3 $0 $218 $44 $50 $11 $2 $85 $5 $10 $0 $22 $11 $37 $85 $125 $20 $33 $7 $1 $31 $32 $13 $360 $1 $1 $0 $111 $44 $22 $54 $80 $3 $5 $64 $447 $9 $161 $80 $41 $50 $15 $5 $76 $29 $74 $79 $3 $0 $5 $135 $10 $85 $30 $43 $42 $11 $50 $5 $0 $74 $3 $20 $140 $85 $2 $1 $44 $12 $10 $6 $171 $16 $41 $20 $5 $89 $10 $78 $2 $140 $12 $752 $15 $5 $3 $0 $3 $220 $8 $37 $126 $3 $18 $26 $18 $4 $20 $225 $1 $25 $3 $58 $17 $15 $2 $15 $24 $0 $121 $25 $70 $30 $55 $6 $1 

In [32]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [33]:
evaluate(grok_4__1_fast, test)

  0%|          | 0/200 [00:00<?, ?it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider L

AuthenticationError: litellm.AuthenticationError: AuthenticationError: XaiException - {"code":"The request does not have valid authentication credentials","error":"No credentials presented. [WKE=unauthenticated:no-credentials]"}

In [36]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [37]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $74 $5 $10 $20 $130 $54 $99 $11 $0 $87 $20 $0 $4 $39 $8 $11 $17 $30 $19 $56 $4 $15 $5 $52 $223 $126 $5 $91 $62 $10 $25 $0 $55 $15 $19 $30 $36 $64 $18 $160 $45 $17 $75 $60 $5 $5 $2 $55 $48 $26 $110 $226 $0 $27 $24 $3 $50 $48 $2 $116 $48 $42 $75 $80 $0 $40 $315 $15 $54 $17 $3 $90 $1 $22 $17 $4 $2 $2 $5 $0 $4 $5 $74 $15 $25 $68 $86 $0 $13 $3 $25 $5 $0 $2 $68 $1 $82 $70 $225 $0 $3 $2 $19 $49 $142 $15 $340 $1 $139 $25 $75 $1 $63 $54 $20 $9 $1 $24 $397 $7 $91 $0 $46 $10 $10 $1 $21 $9 $59 $149 $7 $15 $0 $35 $5 $16 $10 $142 $8 $6 $149 $30 $10 $6 $2 $10 $10 $5 $8 $4 $54 $28 $70 $4 $71 $31 $37 $71 $0 $90 $17 $3 $1 $590 $7 $352 $26 $10 $2 $12 $2 $170 $13 $67 $19 $6 $47 $6 $2 $154 $12 $251 $59 $25 $3 $63 $17 $10 $7 $5 $9 $5 $61 $50 $40 $19 $30 $21 $7 